# Mesin Pencari Skripsi UNJ: dari Cleaning sampai Checkpoint 2

Tugas Project Temu Kembali Informasi, Topik 1 (Pencarian Skripsi dan Jurnal UNJ).

Notebook ini mencakup:

| Bagian | Isi | Untuk |
|---|---|---|
| A | Cleaning dataset | Checkpoint 1 |
| B | Pre-processing (tokenisasi, stopword, stemming Sastrawi) | Checkpoint 2 |
| C | Inverted index buatan sendiri (per field) | Checkpoint 2 |
| D | Model ranking: TF-IDF (kosinus) dan BM25, dengan bobot per field | Checkpoint 2 |
| E | Demo pencarian dan uji kewajaran (sanity check) | Checkpoint 2 |

**Cara pakai:** letakkan `dataset_unj.csv` di folder yang sama dengan notebook, lalu jalankan semua sel dari atas ke bawah. Di Google Colab, hapus tanda `#` pada sel upload di bawah.

**Catatan:** inverted index dan ranking ditulis sendiri (hanya memakai Python standar dan NumPy), sesuai ketentuan tugas. Library pihak ketiga hanya dipakai untuk stemming (Sastrawi).

In [ ]:
# Instalasi (jalankan sekali). Di komputer lokal bisa lewat terminal: pip install pandas numpy PySastrawi
# !pip install -q pandas numpy PySastrawi matplotlib

# Khusus Google Colab: hapus tanda '#' di dua baris berikut lalu pilih dataset_unj.csv
# from google.colab import files
# files.upload()

In [ ]:
import re
import math
import pickle
import heapq
import time
from collections import Counter, defaultdict

import numpy as np
import pandas as pd

INPUT = 'dataset_unj.csv'
CLEAN_OUT = 'dataset_clean.csv'
TOKENS_CACHE = 'tokens_unj.pkl'   # cache hasil stemming (stemming lambat, jangan diulang)
MIN_WORDS = 30                    # abstrak minimal 30 kata

---
# BAGIAN A. Cleaning dataset (Checkpoint 1)

## A1. Muat dan cek data mentah

In [ ]:
df = pd.read_csv(INPUT)
n_awal = len(df)
print('Jumlah baris mentah:', n_awal)
print('\nKolom kosong:'); print(df.isna().sum())
print('\nJenis dokumen:'); print(df['jenis'].value_counts())
print('\nSebaran tahun:'); print(df['tahun'].value_counts().sort_index())
df.head(3)

## A2. Bersihkan

Langkah yang dilakukan:
1. Buang kolom `prodi` (hampir kosong) dan `set` (kode heksadesimal tidak terpakai).
2. Buang sisa **kutipan sitasi** di awal abstrak (memuat nama penulis, melanggar anonimisasi).
3. Buang abstrak non-Latin dan abstrak terlalu pendek.
4. Hapus duplikat judul (simpan yang abstraknya paling panjang).
5. Rapikan kolom `jenis` dan tambahkan `url` (halaman record) serta `sumber`.

In [ ]:
# --- 2. Buang sitasi di awal abstrak ---
CITE_END = re.compile(r'Universitas Negeri Jakarta,?\s*(?:19|20)\d{2}\.')
SKRIPSI_CITE = re.compile(
    r'Skripsi\.\s*.{0,250}?(?:Universitas Negeri Jakarta|State University of Jakarta)'
    r'(?:[,.]?\s*(?:[A-Za-z]+\s+)?(?:19|20)\d{2}\.?|\.)?'
)

def strip_sitasi(abstrak):
    m = SKRIPSI_CITE.search(abstrak[:700])
    if m:
        return abstrak[m.end():].strip()
    m = CITE_END.search(abstrak)
    if m and m.end() < 0.4 * len(abstrak):
        return abstrak[m.end():].strip()
    return abstrak

def non_latin_ratio(text):
    letters = [c for c in str(text) if c.isalpha()]
    if not letters:
        return 1.0
    return sum(ord(c) > 0x024F for c in letters) / len(letters)

# --- 1. Buang kolom tak terpakai ---
df = df.drop(columns=[c for c in ['prodi', 'set'] if c in df.columns])

df['abstrak'] = df['abstrak'].fillna('').map(strip_sitasi)
df['judul'] = df['judul'].fillna('').str.strip()
df['kata_kunci'] = df['kata_kunci'].fillna('')

# --- 3. Abstrak non-Latin dan terlalu pendek ---
m_nl = df['abstrak'].map(non_latin_ratio) > 0.3
print('Abstrak non-Latin dibuang :', int(m_nl.sum()))
df = df[~m_nl]
m_pendek = df['abstrak'].str.split().str.len() < MIN_WORDS
print('Abstrak terlalu pendek    :', int(m_pendek.sum()))
df = df[~m_pendek]

# --- 4. Duplikat judul ---
n_sebelum = len(df)
df['_k'] = df['judul'].str.lower().str.replace(r'[^a-z0-9]+', ' ', regex=True).str.strip()
df['_len'] = df['abstrak'].str.len()
df = df.sort_values('_len', ascending=False).drop_duplicates('_k').sort_index()
df = df.drop(columns=['_k', '_len'])
print('Duplikat judul dibuang    :', n_sebelum - len(df))

# --- 5. Rapikan kolom ---
df['doc_id'] = df['oai_id'].str.extract(r':(\d+)$')[0].astype(int)
df['jenis'] = df['jenis'].str.split(';').str[0].str.strip()      # 'Thesis; NonPeerReviewed' -> 'Thesis'
df['url'] = 'http://repository.unj.ac.id/' + df['doc_id'].astype(str) + '/'
df['sumber'] = 'UNJ repository'
df['tahun'] = pd.to_numeric(df['tahun'], errors='coerce').fillna(0).astype(int)

df = df[['doc_id', 'judul', 'abstrak', 'kata_kunci', 'tahun', 'jenis', 'url', 'sumber']].reset_index(drop=True)
df.to_csv(CLEAN_OUT, index=False, encoding='utf-8-sig')
print(f'\nSebelum: {n_awal} | Sesudah: {len(df)} | Tersimpan di {CLEAN_OUT}')
df.head(3)

In [ ]:
# Ringkasan untuk laporan
print('Jenis dokumen:'); print(df['jenis'].value_counts())
print('\nPanjang abstrak (kata):'); print(df['abstrak'].str.split().str.len().describe().round(1))
print('\nDokumen tanpa kata kunci:', int((df['kata_kunci'] == '').sum()))
df['tahun'].value_counts().sort_index().plot(kind='bar', title='Jumlah dokumen per tahun', figsize=(7, 3))

**Catatan keterbatasan data (untuk laporan):**
- Dokumen terkonsentrasi di tahun 2012-2018, jadi topik terbaru hampir tidak tercakup.
- Hampir semua dokumen adalah skripsi (Thesis) dan non-peer-reviewed.
- `kata_kunci` berisi istilah subjek yang luas, bukan kata kunci dari penulis.
- Nama penulis dan pembimbing tidak disimpan (anonimisasi).

---
# BAGIAN B. Pre-processing (Checkpoint 2)

Tahapannya:
1. **Case folding**: huruf kecil semua.
2. **Tokenisasi**: ambil deretan huruf (angka dan tanda baca dibuang).
3. **Penghapusan stopword**: daftar stopword Bahasa Indonesia dari Sastrawi, ditambah daftar kecil stopword Bahasa Inggris (karena ada abstrak berbahasa Inggris).
4. **Stemming** Bahasa Indonesia dengan Sastrawi.

Stemming Sastrawi lambat, jadi tiap kata unik hanya di-stem **sekali** (disimpan di kamus `stem_cache`) dan hasil akhirnya disimpan ke file cache.

In [ ]:
from Sastrawi.Stemmer.StemmerFactory import StemmerFactory
from Sastrawi.StopWordRemover.StopWordRemoverFactory import StopWordRemoverFactory

stemmer = StemmerFactory().create_stemmer()

EN_STOP = set("""a an the and or of to in on at for from by with without as is are was were be been being this that
these those it its into than then so such not no can could may might will would should has have had do does did
which who whom whose what when where why how about between among over under also their there here more most
other some any each both only own same very""".split())
STOPWORDS = set(StopWordRemoverFactory().get_stop_words()) | EN_STOP
print('Jumlah stopword:', len(STOPWORDS))

stem_cache = {}

def stem_word(w):
    if w not in stem_cache:
        stem_cache[w] = stemmer.stem(w)
    return stem_cache[w]

def preprocess(text):
    """teks mentah -> daftar token (case folding, tokenisasi, stopword, stemming)"""
    text = str(text).lower()
    text = re.sub(r'https?://\S+', ' ', text)
    tokens = re.findall(r'[a-z]+', text)
    tokens = [t for t in tokens if len(t) > 1 and t not in STOPWORDS]
    tokens = [stem_word(t) for t in tokens]
    return [t for t in tokens if len(t) > 1 and t not in STOPWORDS]

contoh = 'Pengaruh Motivasi Belajar terhadap Prestasi Belajar Siswa di Sekolah Dasar'
print(contoh)
print(preprocess(contoh))

In [ ]:
FIELDS = ['judul', 'abstrak', 'kata_kunci']

if True:
    try:
        with open(TOKENS_CACHE, 'rb') as f:
            data = pickle.load(f)
        if data['doc_ids'] == df['doc_id'].tolist():
            tokens = data['tokens']
            print('Memakai cache stemming:', TOKENS_CACHE)
        else:
            raise ValueError('dataset berubah')
    except Exception:
        t0 = time.time()
        tokens = {f: {} for f in FIELDS}
        for i, row in enumerate(df.itertuples(index=False), 1):
            for f in FIELDS:
                tokens[f][row.doc_id] = preprocess(getattr(row, f))
            if i % 500 == 0:
                print(f'  {i}/{len(df)} dokumen diproses ({time.time()-t0:.0f} detik)')
        with open(TOKENS_CACHE, 'wb') as f:
            pickle.dump({'doc_ids': df['doc_id'].tolist(), 'tokens': tokens}, f)
        print(f'Selesai dalam {time.time()-t0:.0f} detik. Cache disimpan ke {TOKENS_CACHE}')

# Contoh hasil pre-processing satu dokumen
r = df.iloc[0]
print('\nJudul asli :', r['judul'])
print('Token judul:', tokens['judul'][r['doc_id']])
print('Token abstrak (20 pertama):', tokens['abstrak'][r['doc_id']][:20])

---
# BAGIAN C. Inverted index buatan sendiri (Checkpoint 2)

Inverted index memetakan **term → daftar dokumen yang memuatnya beserta frekuensinya (tf)**:

```
"ajar" -> {101: 3, 205: 1, 877: 4, ...}      # {doc_id: tf}
```

Dibuat **satu index per field** (judul, abstrak, kata kunci) agar tiap field bisa diberi bobot berbeda. Selain postings, disimpan juga: panjang dokumen per field, dan jumlah dokumen yang memuat term (df).

In [ ]:
class FieldIndex:
    """Inverted index untuk satu field."""

    def __init__(self, tokens_by_doc, n_docs):
        self.N = n_docs                       # jumlah seluruh dokumen koleksi
        self.postings = defaultdict(dict)     # term -> {doc_id: tf}
        self.doc_len = {}                     # doc_id -> jumlah token
        for doc_id, toks in tokens_by_doc.items():
            self.doc_len[doc_id] = len(toks)
            for term, tf in Counter(toks).items():
                self.postings[term][doc_id] = tf
        self.df = {t: len(p) for t, p in self.postings.items()}
        total_len = sum(self.doc_len.values())
        self.avgdl = total_len / max(1, sum(1 for l in self.doc_len.values() if l > 0))
        self._build_tfidf_norms()

    # ---- bobot TF-IDF (bentuk log) ----
    def idf_tfidf(self, term):
        return math.log10(self.N / self.df[term])

    def _build_tfidf_norms(self):
        """Norma vektor tiap dokumen (untuk normalisasi kosinus)."""
        sq = defaultdict(float)
        for term, post in self.postings.items():
            idf = self.idf_tfidf(term)
            for doc_id, tf in post.items():
                w = (1 + math.log10(tf)) * idf
                sq[doc_id] += w * w
        self.norm = {d: math.sqrt(v) for d, v in sq.items()}

    # ---- bobot BM25 ----
    def idf_bm25(self, term):
        df = self.df[term]
        return math.log(1 + (self.N - df + 0.5) / (df + 0.5))


N_DOCS = len(df)
index = {f: FieldIndex(tokens[f], N_DOCS) for f in FIELDS}

for f in FIELDS:
    ix = index[f]
    print(f'{f:11s} | term unik: {len(ix.postings):6d} | rata-rata panjang: {ix.avgdl:6.1f} token')

# Contoh isi inverted index
term = 'ajar'
print(f"\nContoh postings untuk term '{term}' pada field judul:")
print('  df =', index['judul'].df.get(term), '| 5 posting pertama:', list(index['judul'].postings[term].items())[:5])

---
# BAGIAN D. Model ranking (Checkpoint 2)

Dua model dibangun dari inverted index di atas:

**1. TF-IDF dengan kemiripan kosinus (VSM)**
- bobot dokumen: `(1 + log10(tf)) × idf`, idf = `log10(N/df)`
- bobot query: `(1 + log10(tf_query)) × idf`
- skor: kosinus antara vektor query dan vektor dokumen

**2. BM25** (k1 = 1.5, b = 0.75)
- `idf × tf(k1+1) / (tf + k1(1 − b + b·dl/avgdl))`

Skor akhir tiap dokumen adalah **jumlah berbobot skor per field**:
`skor = w_judul × skor_judul + w_kata_kunci × skor_kata_kunci + w_abstrak × skor_abstrak`

Karena skala skor tiap field berbeda, skor tiap field **dinormalkan** dulu dengan membaginya dengan skor maksimum field itu untuk query yang sama (min-max sederhana), sehingga bobot benar-benar mencerminkan prioritas field.

In [ ]:
FIELD_WEIGHTS = {'judul': 3.0, 'kata_kunci': 2.0, 'abstrak': 1.0}   # bisa dituning saat evaluasi
K1, B = 1.5, 0.75


def score_tfidf(ix, q_terms):
    """Skor kosinus TF-IDF untuk satu field. Mengembalikan {doc_id: skor}."""
    qtf = Counter(t for t in q_terms if t in ix.postings)
    if not qtf:
        return {}
    q_w = {t: (1 + math.log10(c)) * ix.idf_tfidf(t) for t, c in qtf.items()}
    q_norm = math.sqrt(sum(w * w for w in q_w.values()))
    if q_norm == 0:
        return {}
    scores = defaultdict(float)
    for t, wq in q_w.items():
        idf = ix.idf_tfidf(t)
        for doc_id, tf in ix.postings[t].items():
            scores[doc_id] += wq * ((1 + math.log10(tf)) * idf)
    return {d: s / (q_norm * ix.norm[d]) for d, s in scores.items() if ix.norm.get(d, 0) > 0}


def score_bm25(ix, q_terms):
    """Skor BM25 untuk satu field. Mengembalikan {doc_id: skor}."""
    scores = defaultdict(float)
    for t in set(q_terms):
        if t not in ix.postings:
            continue
        idf = ix.idf_bm25(t)
        for doc_id, tf in ix.postings[t].items():
            dl = ix.doc_len[doc_id]
            denom = tf + K1 * (1 - B + B * dl / ix.avgdl)
            scores[doc_id] += idf * tf * (K1 + 1) / denom
    return dict(scores)


SCORERS = {'tfidf': score_tfidf, 'bm25': score_bm25}
df_idx = df.set_index('doc_id')


def search(query, model='bm25', k=10, weights=None):
    """Cari query; kembalikan DataFrame top-k beserta skor akhir dan skor per field."""
    weights = weights or FIELD_WEIGHTS
    q_terms = preprocess(query)
    if not q_terms:
        return pd.DataFrame(columns=['doc_id', 'judul', 'tahun', 'skor', 'url'])
    per_field = {}
    for f in FIELDS:
        s = SCORERS[model](index[f], q_terms)
        mx = max(s.values()) if s else 0
        per_field[f] = {d: v / mx for d, v in s.items()} if mx > 0 else {}
    final = defaultdict(float)
    for f in FIELDS:
        for d, v in per_field[f].items():
            final[d] += weights[f] * v
    top = heapq.nlargest(k, final.items(), key=lambda x: (x[1], -x[0]))
    rows = []
    for d, s in top:
        r = df_idx.loc[d]
        rows.append({'doc_id': d, 'judul': r['judul'], 'tahun': r['tahun'], 'skor': round(s, 4),
                     'skor_judul': round(per_field['judul'].get(d, 0), 3),
                     'skor_kata_kunci': round(per_field['kata_kunci'].get(d, 0), 3),
                     'skor_abstrak': round(per_field['abstrak'].get(d, 0), 3),
                     'url': r['url']})
    return pd.DataFrame(rows)


print('Fungsi search() siap. Contoh:')
search('motivasi belajar siswa', model='bm25', k=5)[['doc_id', 'judul', 'skor']]

---
# BAGIAN E. Demo dan uji kewajaran (Checkpoint 2)

## E1. Demo singkat: bandingkan TF-IDF dan BM25

In [ ]:
pd.set_option('display.max_colwidth', 90)
pd.set_option('display.width', 200)

def tampilkan(query, k=5):
    for model in ['tfidf', 'bm25']:
        print(f'\n=== {model.upper()} | query: "{query}" ===')
        hasil = search(query, model=model, k=k)
        for i, r in enumerate(hasil.itertuples(), 1):
            print(f'{i}. [{r.skor:.3f}] ({r.tahun}) {r.judul[:95]}')

tampilkan('motivasi belajar siswa')
tampilkan('pengaruh kepuasan kerja karyawan')
tampilkan('problem based learning')

## E2. Uji dengan 20 query uji (Checkpoint 1)

Tabel berikut membandingkan hasil kedua model: berapa dokumen yang sama di 10 besar (overlap@10) dan apakah dokumen peringkat 1-nya sama. Ini **belum** evaluasi relevansi (itu tahap berikutnya), hanya melihat apakah sistem berjalan untuk semua query.

In [ ]:
QUERIES = [
    'motivasi belajar siswa',
    'pengaruh kepuasan kerja karyawan',
    'pengembangan media pembelajaran IPA',
    'model pembelajaran cooperative learning',
    'keterampilan menulis siswa',
    'pembelajaran bahasa Arab',
    'perusahaan manufaktur yang terdaftar di Bursa Efek Indonesia',
    'faktor yang mempengaruhi keputusan pembelian konsumen',
    'kualitas pelayanan terhadap kepuasan pelanggan',
    'stres kerja dan komitmen organisasi',
    'bimbingan dan konseling siswa',
    'pembelajaran matematika di sekolah dasar',
    'model latihan atlet',
    'pendidikan karakter',
    'kemampuan berpikir kritis',
    'problem based learning',
    'brand image dan citra merek produk',
    'pembelajaran akuntansi di SMK',
    'motivasi belajr siswa SD',          # sengaja salah ketik
    'english learning',
]

rows = []
for i, q in enumerate(QUERIES, 1):
    a = search(q, 'tfidf', k=10)
    b = search(q, 'bm25', k=10)
    overlap = len(set(a['doc_id']) & set(b['doc_id']))
    rows.append({'no': i, 'query': q, 'hasil_tfidf': len(a), 'hasil_bm25': len(b),
                 'overlap@10': overlap,
                 'peringkat1_sama': bool(len(a) and len(b) and a['doc_id'][0] == b['doc_id'][0])})
ringkas = pd.DataFrame(rows)
ringkas

In [ ]:
# Query dengan salah ketik: perhatikan hasilnya (bahan analisis kegagalan & alasan fitur bonus koreksi ejaan)
print('Query benar :', preprocess('motivasi belajar siswa SD'))
print('Query typo  :', preprocess('motivasi belajr siswa SD'))
tampilkan('motivasi belajr siswa SD', k=3)

## E3. Uji kewajaran (sanity check)

Uji sederhana untuk memastikan indexing dan ranking benar: jika **judul sebuah dokumen dipakai sebagai query**, dokumen itu seharusnya muncul di peringkat teratas. Diuji pada 100 dokumen acak.

In [ ]:
rng = np.random.default_rng(42)
sample_ids = rng.choice(df['doc_id'].values, size=100, replace=False)

hasil_uji = {}
for model in ['tfidf', 'bm25']:
    rank1 = rank3 = 0
    for d in sample_ids:
        judul = df_idx.loc[d, 'judul']
        top = search(judul, model=model, k=3)['doc_id'].tolist()
        rank1 += (len(top) > 0 and top[0] == d)
        rank3 += (d in top)
    hasil_uji[model] = (rank1, rank3)
    print(f'{model.upper():6s} | dokumen di peringkat 1: {rank1}/100 | di 3 besar: {rank3}/100')

In [ ]:
# Uji kebenaran index: df di index harus sama dengan hitungan langsung dari token
for f in FIELDS:
    for term in ['ajar', 'siswa', 'kerja']:
        langsung = sum(1 for toks in tokens[f].values() if term in toks)
        assert index[f].df.get(term, 0) == langsung, (f, term)
print('Uji df index: LULUS (cocok dengan hitungan langsung)')

# Efek bobot field: bandingkan bobot judul besar vs seimbang
q = 'motivasi belajar'
print('\nBobot default (judul 3, kata kunci 2, abstrak 1):')
print(search(q, 'bm25', k=3)[['judul', 'skor']].to_string(index=False))
print('\nBobot seimbang (1, 1, 1):')
print(search(q, 'bm25', k=3, weights={'judul': 1, 'kata_kunci': 1, 'abstrak': 1})[['judul', 'skor']].to_string(index=False))

---
# Ringkasan Checkpoint 2

| Syarat Checkpoint 2 | Status |
|---|---|
| Pipeline pre-processing | Bagian B |
| Inverted index (buatan sendiri, per field) | Bagian C |
| Minimal satu model ranking berjalan | Bagian D: TF-IDF dan BM25 (dua model sudah siap) |
| Demo singkat | Bagian E |

**Langkah berikutnya (setelah Checkpoint 2):**
1. Fitur **dokumen serupa** (kemiripan kosinus antar dokumen).
2. **Antarmuka web** (Streamlit) yang menampilkan hasil beserta skornya.
3. **Evaluasi**: penilaian relevansi (anggota kelompok + satu penilai luar), hitung Precision@k, MAP, nDCG; tuning bobot field.
4. Bonus: filter tahun, koreksi ejaan berbasis k-gram.

**Yang dicatat untuk laporan:** penggunaan alat AI, sumber luar (Sastrawi, OAI-PMH repositori UNJ), dan kontribusi tiap anggota.